In [1]:
import itertools
import shutil
from pathlib import Path
import json

import faiss
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

from torchvision import transforms as T
from opr.datasets.itlp import ITLPCampus
#from opr.models.place_recognition import MinkLoc3D
from mmpr.inference import PlaceRecognitionPipeline, FaissFlatIndex, SequencePlaceRecognitionPipeline, PlaceRecognitionRerankPipeline

from gsloc.inference.pr_infer import PRInferencer, PRRerankInferencer
from gsloc.models import opr_graph_extention as network
# from opr.pipelines.place_recognition import PlaceRecognitionPipeline

from mmpr.models import MegaLoc
from gsloc.datasets import ThreeRScan

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-04-30 06:56:37.929 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


In [2]:
dataset_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan"
test_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan")
index1_path = test_dir / "index1"
index2_path = test_dir / "index2"
query_cache_path = test_dir / "query_cache"
bench_report_dir = test_dir / "seq_benchmark_report"
graph_dir = "SceneGraphs_real_classes_pt_compact"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"

In [3]:
from torchvision.transforms import functional as F

image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True)
])

In [4]:
three_rscan_ds = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=index1_path,
    rebuild_meta=False,  # meta.parquet already built
    # limit=20000,
    image_transform=image_transform_fn,
    save_meta=False,
    scene_filter_mode="listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
    graph_feat_dim = 4,
    graph_edge_attr_dim = 10,
    graph_rotate = True,
    graph_dir=graph_dir,
    edge_normalizer_path=edge_normalizer_path,
)

In [5]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=128,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=256,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4
    ).to(device)

model1  = network.OPR_MultiModalVPRGraphEncoder(
    graph_encoder=OPR_GAT_graph_encoder,
    image_encoder=None,
    image_out_dim=8448,
    graph_out_dim=256,
    fusion_dim=8448,
    normalize=True,
    graph_fusion_scale=0.05,
    freeze_image_encoder=True,
    mode="graph")

missing, unexpected = model1.load_state_dict(ckpt["model_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

model1.to(device)
model1.eval()

OPR_MultiModalVPRGraphEncoder(
  (graph_encoder): OPR_GATGraphEncoder(
    (edge_cont_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (edge_lbl_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (node_emb): Embedding(529, 128)
    (edge_emb): Embedding(41, 128)
    (edge_cont_mlp): Sequential(
      (0): Linear(in_features=10, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_gate): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): Sigmoid()
    )
    (edge_label_proj): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=512, bias=True)
    )
    (edge_fuse): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (

In [6]:
model2 = MegaLoc()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2.to(device)
model2.eval()

Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


MegaLoc(
  (model): MegaLocModel(
    (backbone): DINOv2(
      (model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (blocks): ModuleList(
          (0-11): 12 x NestedTensorBlock(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (attn): MemEffAttention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=768, out_features=3072, bias=True)
              (act): GELU(approximate='none')
              (fc2): Linear(in_features=3072, out_features=768, 

In [7]:
index1 = FaissFlatIndex.generate(
    directory=index1_path,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model1,
    rebuild_meta=False,
    rebuild_descriptors=False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index1_path}")
print(f"Index size: {index1.size()}, dim: {index1.dim()} metric: {index1.metric()}")

2026-04-30 06:56:43.197 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-04-30 06:56:43.197 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-04-30 06:56:43.200 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/index1


Index created at /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/index1
Index size: 9449, dim: 256 metric: l2


In [8]:
index2 = FaissFlatIndex.generate(
    directory=index2_path,
    dataset=three_rscan_ds,
    dataloader=None,
    model=model2,
    rebuild_meta=False,
    rebuild_descriptors=False,
    batch_size = 24,
    num_workers = 6,
    shuffle = False,
    metric = "l2", # can be also "ip" - inner product
    version = 1)
print(f"Index created at {index2_path}")
print(f"Index size: {index2.size()}, dim: {index2.dim()} metric: {index2.metric()}")

2026-04-30 06:56:43.239 | INFO     | mmpr.inference.index:generate:438 - Using existing meta.parquet
2026-04-30 06:56:43.239 | INFO     | mmpr.inference.index:generate:467 - Using existing descriptors.npy
2026-04-30 06:56:43.297 | INFO     | mmpr.inference.index:generate:485 - schema.json file was saved in /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/index2


Index created at /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/index2
Index size: 9449, dim: 8448 metric: l2


In [15]:
pipeline = PlaceRecognitionRerankPipeline(
    index1=index1,
    index2=index2,
    model1=model1,
    model2=model2,
    device="cuda",
)
base_pipeline = PlaceRecognitionPipeline(
    index=index2,
    model=model2,
    device="cuda",
)

In [10]:
three_rscan_q = ThreeRScan(
    dataset_root=dataset_path,
    meta_path=query_cache_path,
    # save_meta=True,
    rebuild_meta=False,
    # limit=10000,
    image_transform=image_transform_fn,
    scene_filter_mode="same_room_excluding_listed",
    scene_list_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt",
    room_json_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json",
    graph_feat_dim = 4,
    graph_edge_attr_dim = 10,
    graph_rotate = True,
    graph_dir=graph_dir,
    edge_normalizer_path=edge_normalizer_path,
)

In [37]:
inferencer = PRRerankInferencer(
    pr_rerank_pipeline=pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=250,
    device="cuda"
)

base_inferencer = PRInferencer(
    pr_pipeline=base_pipeline,
    query_dataset=three_rscan_q,
    batch_size=16,
    num_workers=4,
    query_cache_dir=query_cache_path,
    k=25,
    device="cuda"
)

In [27]:
frames = inferencer.run(rebuild_query_descriptors=True)
inferencer.save(query_cache_path / "frames50.npz", frames=frames)

Compute descriptors + PR cache:  12%|█▏        | 152/1314 [00:39<05:02,  3.85it/s]


KeyboardInterrupt: 

In [28]:
frames = inferencer.run(rebuild_query_descriptors=True)
inferencer.save(query_cache_path / "frames150.npz", frames=frames)

Compute descriptors + PR cache: 100%|██████████| 1314/1314 [05:45<00:00,  3.81it/s]
2026-04-30 08:06:52.638 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/query_cache/meta.parquet


In [34]:
frames = inferencer.run(rebuild_query_descriptors=True)
inferencer.save(query_cache_path / "frames200.npz", frames=frames)

Compute descriptors + PR cache: 100%|██████████| 1314/1314 [05:52<00:00,  3.72it/s]
2026-04-30 08:42:23.811 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/query_cache/meta.parquet


In [38]:
frames = inferencer.run(rebuild_query_descriptors=True)
inferencer.save(query_cache_path / "frames250.npz", frames=frames)

Compute descriptors + PR cache: 100%|██████████| 1314/1314 [06:12<00:00,  3.53it/s]
2026-04-30 08:52:01.706 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/query_cache/meta.parquet


In [47]:
base_frames = base_inferencer.run(rebuild_query_descriptors=True)
base_inferencer.save(query_cache_path / "base_frames.npz", frames=base_frames)

Compute descriptors + PR cache: 100%|██████████| 1314/1314 [07:39<00:00,  2.86it/s]
2026-04-30 13:33:17.882 | INFO     | gsloc.datasets.pr_dataset:save_meta_parquet:51 - Wrote 21,013 rows to /home/kartashov_ga/projects/GSLoc/data/tests/30-04-25/Fusion/3rscan/query_cache/meta.parquet


# k = 100

In [13]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 5898.46it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18125,0.862561,86.256127
1,5,21013,19094,0.908676,90.867558
2,10,21013,19434,0.924856,92.485604
3,25,21013,19843,0.944320,94.432018


In [14]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 180
    },
    include_per_query=False
)

  0%|          | 0/21013 [00:00<?, ?it/s]/home/kartashov_ga/projects/GSLoc/src/gsloc/datasets/three_rscan.py:495: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  pose_a = torch.as_tensor(pose_a)
100%|██████████| 21013/21013 [00:14<00:00, 1418.07it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,10539,0.501547,50.154666
1,5,21013,12492,0.594489,59.448913
2,10,21013,13469,0.640984,64.098415
3,25,21013,14870,0.707657,70.765716


# MegaLoc

In [18]:
base_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:02<00:00, 7038.62it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18608,0.885547,88.554704
1,5,21013,19636,0.934469,93.446914
2,10,21013,20014,0.952458,95.245800
3,25,21013,20434,0.972446,97.244563


In [19]:
base_inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 180
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:15<00:00, 1390.07it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,10806,0.514253,51.425308
1,5,21013,12780,0.608195,60.819493
2,10,21013,13955,0.664113,66.411269
3,25,21013,15721,0.748156,74.815590


# k = 50

In [22]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 5442.32it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,17873,0.850569,85.056870
1,5,21013,18790,0.894208,89.420835
2,10,21013,19150,0.911341,91.134060
3,25,21013,19501,0.928045,92.804454


In [23]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 180
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:14<00:00, 1431.82it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,10416,0.495693,49.569314
1,5,21013,12242,0.582592,58.259173
2,10,21013,13264,0.631228,63.122829
3,25,21013,14434,0.686908,68.690810


# k = 150 

In [31]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 6097.34it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18246,0.868320,86.831961
1,5,21013,19222,0.914767,91.476705
2,10,21013,19577,0.931661,93.166135
3,25,21013,19997,0.951649,95.164898


In [32]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 180
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:14<00:00, 1426.43it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,10619,0.505354,50.535383
1,5,21013,12585,0.598915,59.891496
2,10,21013,13582,0.646362,64.636178
3,25,21013,15100,0.718603,71.860277


# k = 200

In [35]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 6133.88it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18363,0.873888,87.388759
1,5,21013,19337,0.920240,92.023985
2,10,21013,19643,0.934802,93.480227
3,25,21013,20048,0.954076,95.407605


In [36]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 180
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:14<00:00, 1400.88it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,10653,0.506972,50.697187
1,5,21013,12651,0.602056,60.205587
2,10,21013,13651,0.649645,64.964546
3,25,21013,15175,0.722172,72.217199


In [39]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "room"
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:03<00:00, 6225.22it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,18427,0.876933,87.693333
1,5,21013,19412,0.923809,92.380907
2,10,21013,19712,0.938086,93.808595
3,25,21013,20116,0.957312,95.731214


# k = 250

In [40]:
inferencer.build_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    ks=[1, 5, 10, 25],
    similarity_kwargs={
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 180
    },
    include_per_query=False
)

100%|██████████| 21013/21013 [00:14<00:00, 1407.69it/s]


,k,num_queries,num_correct,recall_at_k,recall_at_k_percent
0,1,21013,10682,0.508352,50.835197
1,5,21013,12696,0.604197,60.419740
2,10,21013,13714,0.652644,65.264360
3,25,21013,15240,0.725265,72.526531


In [42]:
bench_report_dir = test_dir / "seq_benchmark_report"
curr_report_dir = bench_report_dir / "room_k25_batch-std_rerank250"

result_df = inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  9%|▉         | 1/11 [01:32<15:28, 92.85s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 18%|█▊        | 2/11 [03:44<17:22, 115.81s/it]

...

 82%|████████▏ | 9/11 [51:36<16:57, 508.67s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 91%|█████████ | 10/11 [1:03:31<09:32, 572.29s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


100%|██████████| 11/11 [1:15:55<00:00, 414.17s/it]


In [43]:
result_df

,w,auc_pr,f1_max,recall_at_1,recall_at_1_std,recall_at_5,recall_at_5_std,recall_at_10,recall_at_10_std,recall_at_25,recall_at_25_std,num_valid,num_total
0,1,0.921708,0.848886,0.876933,0.034476,0.923809,0.026475,0.938086,0.023548,0.957312,0.020032,21013,21013
1,2,0.907099,0.829058,0.895160,0.033226,0.936896,0.026654,0.949650,0.023961,0.964879,0.019139,21013,21013
2,3,0.898672,0.818273,0.906629,0.031261,0.944082,0.024834,0.956075,0.021327,0.970352,0.017582,21013,21013
3,5,0.888441,0.806015,0.919907,0.027114,0.955409,0.020405,0.965973,0.017847,0.977680,0.014880,21013,21013
4,7,0.882333,0.798969,0.929710,0.025772,0.962499,0.018932,0.972303,0.015713,0.982011,0.013134,21013,21013
5,10,0.876200,0.791837,0.938752,0.023218,0.969067,0.016951,0.977443,0.014370,0.986294,0.010484,21013,21013
6,15,0.870071,0.784846,0.947556,0.020997,0.975967,0.014441,0.982439,0.012396,0.991006,0.009786,21013,21013
7,20,0.867875,0.782407,0.952553,0.021552,0.978680,0.014668,0.985771,0.011575,0.993337,0.008507,21013,21013
8,25,0.869359,0.783445,0.955742,0.021185,0.980012,0.013736,0.987865,0.010860,0.994527,0.007455,21013,21013
9,30,0.873092,0.786625,0.956551,0.020770,0.980917,0.014277,0.989435,0.009956,0.994956,0.007473,21013,21013


In [44]:
def plot_metrics_vs_window_with_stats(
    summary_df,
    summary_all,
    metrics = ("auc_pr", "f1_max", "recall_at_1", "recall_at_5", "recall_at_10", "recall_at_25"),
):
    """Plot per-map metrics vs w and overlay cross-map mean and weighted mean.

    Args:
        summary_df: DataFrame for a single map (has columns 'w', metrics, and optional '<metric>_std').
        summary_all: Concatenated DataFrame across maps with columns 'w', 'query_track', 'num_valid', and metrics.
        metrics: metric names to visualize.
    Returns:
        dict metric -> plotly figure
    """
    figs = {}
    df = summary_df.sort_values("w").reset_index(drop=True)
    map_name = "all"

    for m in metrics:
        if m not in df.columns:
            continue

        std_col = f"{m}_std"
        has_std = std_col in df.columns

        line_kwargs = {
            "x": "w",
            "y": m,
            "title": f"{map_name}: {m} vs sequence length (w)",
            "markers": True,
        }
        if has_std:
            line_kwargs["error_y"] = std_col

        fig = px.line(df, **line_kwargs)
        fig.update_layout(xaxis_title="sequence length (max_window)", yaxis_title=m)

        # Highlight maximum point on per-map line
        try:
            idx_max = df[m].astype(float).idxmax()
            w_star = int(df.loc[idx_max, "w"])  # sequence length at max
            y_star = float(df.loc[idx_max, m])
            fig.add_trace(
                go.Scatter(x=[w_star], y=[y_star], mode="markers", marker=dict(color="red", size=10), name="max", showlegend=False)
            )
            try:
                fig.add_vline(x=w_star, line_dash="dash", line_color="red")
            except Exception:
                fig.add_shape(type="line", x0=w_star, x1=w_star, y0=min(df[m].astype(float)), y1=max(df[m].astype(float)), line=dict(color="red", dash="dash"))
            fig.add_annotation(x=w_star, y=y_star, text=f"w={w_star}, {m}={y_star:.4f}", showarrow=True, arrowhead=2, ax=40, ay=-40)
        except Exception:
            pass

        # Overlay weighted mean across maps (weights = num_valid per map)
        try:
            wmean_series = (
                summary_all
                .groupby("w")
                .apply(lambda g: float(np.average(g[m].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                .reset_index(name=m)
            )

            weighted_mean_kwargs = {
                "x": wmean_series["w"],
                "y": wmean_series[m].astype(float),
                "mode": "lines",
                "name": "weighted mean",
                "line": dict(color="purple", dash="dot"),
                "showlegend": True,
            }

            if std_col in summary_all.columns:
                wstd_series = (
                    summary_all
                    .groupby("w")
                    .apply(lambda g: float(np.average(g[std_col].astype(float), weights=g["num_valid"].astype(float))), include_groups=False)
                    .reset_index(name=std_col)
                )
                wmean_series = wmean_series.merge(wstd_series, on="w", how="left")
                weighted_mean_kwargs["error_y"] = dict(type="data", array=wmean_series[std_col].astype(float), visible=True)

            fig.add_trace(go.Scatter(**weighted_mean_kwargs))
        except Exception:
            pass

        figs[m] = fig
        fig.show()
    return figs


In [45]:
plot_metrics_vs_window_with_stats(result_df, result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('GlN1gaF+7T+90A6u8wbtP4rFUmDswe' ... 'zJ0es//PLs113w6z+dtce8XhbsPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [1],
               'y': [0.9217078712923439]},
              {'line': {'color': 'purple', 'dash': 'dot'},
            

In [48]:
bench_report_dir = test_dir / "seq_benchmark_report"
curr_report_dir = bench_report_dir / "room_k25_batch-std_base"

base_result_df = base_inferencer.build_sequence_recall_benchmark_report(
    database_dataset=three_rscan_ds,
    similarity_kwargs={
        "mode": "room",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
        },
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    per_frame_k_used=25,
    save_dir=curr_report_dir,
    std_mode="global",
    scene_df_field="scene",
)

  0%|          | 0/11 [00:00<?, ?it/s]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


  9%|▉         | 1/11 [01:31<15:10, 91.10s/it]

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started
fused rankings preparation started


 18%|█▊        | 2/11 [03:02<13:40, 91.21s/it]

...

rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started


 91%|█████████ | 10/11 [15:11<01:31, 91.10s/it]

fused rankings preparation started
rankings creation started
ranking iteration started
recall@k calculation started
micro curves calculation started


100%|██████████| 11/11 [16:42<00:00, 91.13s/it]

fused rankings preparation started


In [49]:
base_result_df

,w,auc_pr,f1_max,recall_at_1,recall_at_1_std,recall_at_5,recall_at_5_std,recall_at_10,recall_at_10_std,recall_at_25,recall_at_25_std,num_valid,num_total
0,1,0.920085,0.868162,0.885547,0.035180,0.934469,0.025987,0.952458,0.021477,0.972446,0.016858,21013,21013
1,2,0.926391,0.881454,0.896207,0.030160,0.942512,0.024200,0.958121,0.020680,0.976062,0.016443,21013,21013
2,3,0.931120,0.890320,0.902489,0.030329,0.947128,0.023558,0.962357,0.019367,0.979203,0.015108,21013,21013
3,5,0.938111,0.902464,0.910865,0.028149,0.953648,0.021035,0.968638,0.017379,0.982630,0.013539,21013,21013
4,7,0.942880,0.911378,0.918812,0.029260,0.958312,0.021860,0.973112,0.017584,0.985295,0.013184,21013,21013
5,10,0.947674,0.920006,0.926617,0.026428,0.964022,0.019040,0.976110,0.014966,0.988626,0.010837,21013,21013
6,15,0.952330,0.929031,0.934850,0.024961,0.969781,0.017621,0.979632,0.013519,0.991291,0.008936,21013,21013
7,20,0.955185,0.934350,0.938895,0.023402,0.972398,0.016552,0.982154,0.013341,0.992481,0.009110,21013,21013
8,25,0.957093,0.938053,0.941370,0.023608,0.974254,0.016136,0.983153,0.012656,0.993385,0.008451,21013,21013
9,30,0.958265,0.940661,0.942226,0.023442,0.975444,0.014602,0.984057,0.011790,0.994051,0.007604,21013,21013


In [50]:
plot_metrics_vs_window_with_stats(base_result_df, base_result_df)

{'auc_pr': Figure({
     'data': [{'hovertemplate': 'w=%{x}<br>auc_pr=%{y}<extra></extra>',
               'legendgroup': '',
               'line': {'color': '#636efa', 'dash': 'solid'},
               'marker': {'symbol': 'circle'},
               'mode': 'lines+markers',
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'type': 'scatter',
               'x': {'bdata': 'AQIDBQcKDxQZHiM=', 'dtype': 'i1'},
               'xaxis': 'x',
               'y': {'bdata': ('mSNzsVVx7T9qhpSP/qTtP1KhoXS8y+' ... 'qBoO4/L8nxBBuq7j+xlOp637DuPw=='),
                     'dtype': 'f8'},
               'yaxis': 'y'},
              {'marker': {'color': 'red', 'size': 10},
               'mode': 'markers',
               'name': 'max',
               'showlegend': False,
               'type': 'scatter',
               'x': [35],
               'y': [0.9590909386357876]},
              {'line': {'color': 'purple', 'dash': 'dot'},
           